# Pipeline Huấn luyện SCAR

## Bước 1: Kiểm tra GPU & Kết nối Google Drive


In [1]:
# 1. Kiểm tra cấu hình GPU
!nvidia-smi

# 2. Kết nối Google Drive để lưu trữ và tải dữ liệu/weights
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')


Wed Sep  9 10:04:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Bước 2: Clone SCAR


In [2]:
%cd /content
!rm -rf SCAR
!git clone https://github.com/thanhquan123hi1/SCAR.git
%cd /content/SCAR
print("Thư mục làm việc hiện tại:", os.getcwd())


/content
Cloning into 'SCAR'...
remote: Enumerating objects: 398, done.
remote: Counting objects: 100% (398/398), done.
remote: Compressing objects: 100% (254/254), done.
remote: Total 398 (delta 161), reused 358 (delta 121), pack-reused 0 (from 0)
Receiving objects: 100% (398/398), 482.10 KiB | 14.61 MiB/s, done.
Resolving deltas: 100% (161/161), done.
/content/SCAR
Thư mục làm việc hiện tại: /content/SCAR


## Bước 3: Cài đặt thư viện

In [3]:
!pip install -r requirements.txt
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.7 MB/s eta 0:00:00
PyTorch Version: 2.11.0+cu128
CUDA Available: True
Device Name: NVIDIA A100-SXM4-80GB


## Bước 4: Chuẩn bị Pretrained Weights (`R50-ViT-B_16.npz`)

In [4]:
# Tạo thư mục chứa weights
!mkdir -p /content/SCAR/model/vit_checkpoint/imagenet21k

# CÁCH 1: Copy từ Google Drive cá nhân (Chuẩn đường dẫn Drive của bạn)
drive_vit_path = "/content/drive/MyDrive/NCKH_V2/i_mmseg/ViT_checkpoint/R50-ViT-B_16.npz"

if os.path.exists(drive_vit_path):
    !cp "$drive_vit_path" /content/SCAR/model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz
    print("✅ Đã copy R50-ViT-B_16.npz từ Google Drive cá nhân!")
else:
    print("⚠️ Không tìm thấy file trong Drive cá nhân. Đang tự động tải từ server Google ViT...")
    # CÁCH 2: Tải trực tiếp từ server Google ViT ImageNet21k
    !wget -c -O /content/SCAR/model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz

# Kiểm tra file đã sẵn sàng
!ls -lh /content/SCAR/model/vit_checkpoint/imagenet21k/


✅ Đã copy R50-ViT-B_16.npz từ Google Drive cá nhân!
total 440M
-r-------- 1 root root 440M Sep  9 10:05 R50-ViT-B_16.npz


## Bước 5: Chuẩn bị Dataset MyoPS380
Sao chép thư mục dataset chuẩn từ Google Drive (`/content/drive/MyDrive/NCKH_V2/i_mmseg/MyoPS380_dataset`) vào môi trường Colab.


In [5]:
# Tạo thư mục MyoPS380_dataset trong project
!mkdir -p /content/SCAR/MyoPS380_dataset

# Đường dẫn thư mục dataset trên Google Drive cá nhân (Chuẩn đường dẫn của bạn)
drive_dataset_dir = "/content/drive/MyDrive/NCKH_V2/i_mmseg/MyoPS380_dataset"

# Thư mục đích trong project trên Colab
target_dataset_dir = "/content/SCAR/MyoPS380_dataset"

if os.path.exists(drive_dataset_dir):
    print(f"Đang sao chép dataset từ Drive: {drive_dataset_dir}...")
    os.makedirs(target_dataset_dir, exist_ok=True)
    !cp -r "$drive_dataset_dir"/* "$target_dataset_dir"/
    print("Sao chép dataset thành công!")
    print("Các thư mục hiện có:")
    !ls -la "$target_dataset_dir"
else:
    print(f"Không tìm thấy thư mục: {drive_dataset_dir}")
    print("Hãy kiểm tra lại xem thư mục MyoPS380_dataset nằm ở thư mục nào trong Drive của bạn.")


Đang sao chép dataset từ Drive: /content/drive/MyDrive/NCKH_V2/i_mmseg/MyoPS380_dataset...
✅ Sao chép dataset thành công!
Các thư mục hiện có:
total 16
drwxr-xr-x 4 root root 4096 Sep  9 10:10 .
drwxr-xr-x 8 root root 4096 Sep  9 10:05 ..
dr-x------ 5 root root 4096 Sep  9 10:09 Processed_data
dr-x------ 6 root root 4096 Sep  9 10:10 Raw_data


## Bước 6: Train

In [6]:
!python preprocessing/build_splits.py \
    --data-root /content/SCAR/MyoPS380_dataset/Processed_data \
    --list-dir /content/SCAR/data/processed/splits \
    --test-list preprocessing/splits/test_vol.txt \
    --seed 1234

{
  "patient_counts": {
    "train": 243,
    "val": 61,
    "test_vol": 76
  },
  "sample_counts": {
    "train": 1578,
    "val": 394,
    "val_vol": 61,
    "test_vol": 76
  },
  "list_dir": "/content/SCAR/data/processed/splits"
}


In [ ]:
!python train.py \
    --config training/config/models/cmspa_net.yaml \
    --data-root MyoPS380_dataset/Processed_data \
    --list-dir data/processed/splits \
    --run-root runs \
    --pretrained model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz \
    --batch-size 16 \
    --lr 0.0003 \
    --seed 1234 \
    --amp auto \
    --label-order canonical

2026-09-09 10:11:30,574 | INFO | Device cuda | AMP torch.bfloat16 | parameters 64,525,984 | train/val slices 1578/394 | effective batch <= 16
2026-09-09 10:11:30,575 | INFO | Selection: validation patient-mean scar/exclusive-edema Dice (avg_pathology_dice); early stopping patience=35
2026-09-09 10:11:31.305692: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-09 10:11:31.377370: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-09 10:11:44,893 | INFO | train batch 20/99 | loss 0.94689 | ce 0.92157 | dice_los

## Bước 7: Đánh giá mô hình trên tập Test Volume (3D Testing)
Thực hiện dự đoán và tính toán đầy đủ các chỉ số định lượng (Dice, HD95, ASD) trên toàn bộ các ca bệnh kiểm thử 3D trong `test_vol.txt`, đồng thời xuất các file dự đoán NIfTI để trực quan hóa giải phẫu.


In [ ]:
!python test.py \
    --checkpoint /content/SCAR/runs/CMSPA-Net_seed1234_10h11/best.pth \
    --data-root /content/SCAR/MyoPS380_dataset/Processed_data \
    --split test_vol \
    --device cuda \
    --amp auto \
    --batch-size 8 \
    --save-predictions

## Bước 8: Trực quan hóa Biểu đồ Huấn luyện
Vẽ biểu đồ Loss (Train / Val) và chỉ số hiệu năng phân đoạn (Dice Score, IoU) qua từng epoch từ `metrics.csv`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

metrics_path = Path("/content/SCAR/runs/M3/metrics.csv")
if not metrics_path.is_file():
    metrics_path = Path("/content/drive/MyDrive/NCKH_V2/SCAR_runs/M3/metrics.csv")

if metrics_path.is_file():
    metrics = pd.read_csv(metrics_path)
    display(metrics.tail())

    plt.figure(figsize=(14, 5))

    # Đồ thị Loss
    plt.subplot(1, 2, 1)
    plt.plot(metrics["epoch"], metrics["train/loss"], label="Train Loss", color="blue")
    if "val/loss" in metrics.columns:
        plt.plot(metrics["epoch"], metrics["val/loss"], label="Val Loss", color="orange")
    plt.title("Đường cong mất mát (Loss Curve)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    # Đồ thị Dice & IoU
    plt.subplot(1, 2, 2)
    if "val/mean_dice" in metrics.columns:
        plt.plot(metrics["epoch"], metrics["val/mean_dice"], label="Val Mean Dice", color="green")
    if "val/mean_iou" in metrics.columns:
        plt.plot(metrics["epoch"], metrics["val/mean_iou"], label="Val Mean IoU", color="purple")
    plt.title("Hiệu năng phân đoạn (Validation Metrics)")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()
else:
    print("Chưa tìm thấy file metrics.csv tại:", metrics_path)


## Bước 9: Sao lưu Checkpoints & Kết quả vào Google Drive


In [ ]:
backup_drive_dir = Path("/content/drive/MyDrive/Quan")
backup_drive_dir.mkdir(parents=True, exist_ok=True)
# Tự động tìm thư mục run mới nhất trong runs/
run_dirs = sorted([p for p in Path("/content/SCAR/runs").iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not run_dirs:
    raise FileNotFoundError("Chưa có thư mục run nào trong /content/SCAR/runs")
latest_run = run_dirs[-1]
print(f"Đang sao lưu thư mục: {latest_run.name} ...")
# Copy nguyên thư mục vào Drive
!cp -r "$latest_run" "$backup_drive_dir/"
print(f"Đã sao lưu thành công vào: {backup_drive_dir / latest_run.name}")
!ls -lh "$backup_drive_dir/{latest_run.name}"

In [ ]:
!zip -r /content/SCAR/runs/M3.zip /content/SCAR/runs/M3

  adding: content/SCAR/runs/M3/ (stored 0%)
  adding: content/SCAR/runs/M3/splits/ (stored 0%)
  adding: content/SCAR/runs/M3/splits/val.txt (deflated 87%)
  adding: content/SCAR/runs/M3/splits/test_vol.txt (deflated 79%)
  adding: content/SCAR/runs/M3/splits/split_metadata.json (deflated 85%)
  adding: content/SCAR/runs/M3/splits/val_vol.txt (deflated 74%)
  adding: content/SCAR/runs/M3/splits/train.txt (deflated 87%)
  adding: content/SCAR/runs/M3/train.log (deflated 79%)
  adding: content/SCAR/runs/M3/last.pth (deflated 7%)
  adding: content/SCAR/runs/M3/tensorboard/ (stored 0%)
  adding: content/SCAR/runs/M3/tensorboard/epochs/ (stored 0%)
  adding: content/SCAR/runs/M3/tensorboard/epochs/events.out.tfevents.1788794166.5f4e3e889432.12561.0 (deflated 71%)
  adding: content/SCAR/runs/M3/tensorboard/updates/ (stored 0%)
  adding: content/SCAR/runs/M3/tensorboard/updates/events.out.tfevents.1788794166.5f4e3e889432.12561.1 (deflated 70%)
  adding: content/SCAR/runs/M3/summary.json (defl

In [ ]:
!cp -r /content/SCAR/runs/M3 /content/drive/MyDrive/